In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Thesis_Repository/Final_Google_Drive') # change directory to the current working directory

Mounted at /content/drive


## 1. Settings

In [ ]:
import json
import re
import random
import os
from pathlib import Path

import yaml

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

from transformers import (
    AutoTokenizer,
    # AutoModelForSequenceClassification,
    AutoModelForMaskedLM, # for maskd language model training
    # DataCollatorWithPadding,
    DataCollatorForLanguageModeling, # for masked language model training
    TrainingArguments,
    Trainer,
)

PRETRAINING_CONFIG_PATH = Path("./configs/further_pretraining.yaml")
MATHTUTOR_EEDI_ALL_TURN_DATA_PATH = Path("/content/drive/MyDrive/Thesis_Repository/Final_Google_Drive/data/pretraining_data/mathtutormr_eedi__preceding_1_turn__with_speaker_info.csv")
CHECKPOINT_OUT_DIR = "./further_pretraining_model/further_pretraining__preceding_1__with_speaker_info/checkpoints"
MODEL_OUT_DIR = "./further_pretraining_model/further_pretraining__preceding_1__with_speaker_info/best_model"

MODEL_ID = "FacebookAI/roberta-base"
SEED = 42


# Set Seed
def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.mps.manual_seed(seed)
    except Exception:
        pass

set_all_seeds(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 2. Load dataset

In [ ]:
df = pd.read_csv(MATHTUTOR_EEDI_ALL_TURN_DATA_PATH)

In [ ]:
df.head()

,text
0,S: 36? </s>T: that doesnt sound right thonk i ...
1,S: yeah </s>T: integral of 1 x from -1 to 1 i ...
2,"S: alright, theyre suuper messy though and in ..."
3,S: uhh i think u got it wrong but ok ill solve...
4,S: ohhhhh thanks sneaky </s>T: no worries mate


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117171 entries, 0 to 117170
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    117171 non-null  object
dtypes: object(1)
memory usage: 915.5+ KB


## 3. Train, Validation, Test split

The unlabeled corpus was split into training, validation, and test sets using a 98:1:1 ratio. Since the purpose of the corpus was continued masked-language-model pretraining rather than supervised evaluation, most of the data was retained for training. The validation and test sets still contained approximately 1,100 dialogue samples each, providing a sufficiently large number of token-level MLM predictions for monitoring validation loss and reporting held-out MLM performance.

The purpose of further pretraining is to improve the classification performance on the downstream task ( Don't Stop Pretraining : Adapt Language Models to Domain and Tasks)

The performance is evaluated on the downstream classification task.

In [ ]:
# 99 : 1

train_df, val_df = train_test_split(
    df,
    test_size=0.01,
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


In [ ]:
print("[Split sizes]")
print("train:", len(train_df))
print("val:", len(val_df))

[Split sizes]
train: 115999
val: 1172


## 3. Batch Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,  truncation=True, max_length=512, special_tokens=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        add_special_tokens=True, # include <s> and </s> tokens in the beginning and end of the text
    )


def build_dataset(split_data) -> Dataset:
    ds = Dataset.from_pandas(
        split_data,
        preserve_index=False,
    )

    ds = ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
    )

    return ds

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
from collections import Counter

train_ds = build_dataset(train_df)
val_ds = build_dataset(val_df)
# test_ds = build_dataset(test_df)

print("\n[Dataset columns]")
print("train:", train_ds.column_names)
print("val:", val_ds.column_names)
# print("test:", test_ds.column_names)

print("\n[One tokenized example]")
print(train_ds[0].keys())
print(train_ds[0]['input_ids'])
print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))


Map:   0%|          | 0/115999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1172 [00:00<?, ? examples/s]


[Dataset columns]
train: ['input_ids', 'attention_mask']
val: ['input_ids', 'attention_mask']

[One tokenized example]
dict_keys(['input_ids', 'attention_mask'])
[0, 104, 35, 1368, 5471, 172, 2085, 939, 64, 464, 24, 7, 10, 14313, 2088, 141, 251, 24, 74, 185, 7, 14313, 2088, 53, 172, 24, 67, 7971, 15, 5, 17590, 1437, 2, 565, 35, 187, 63, 10, 481, 12175, 6, 2340, 82, 3334, 40067, 6008, 44949, 281, 1780, 9, 5, 1762, 9, 935, 4, 98, 52, 64, 67, 3724, 66, 5, 514, 1164, 142, 14, 74, 3334, 3549, 201, 350, 4, 98, 74, 3999, 24, 95, 28, 10, 948, 9, 141, 1769, 42, 2173, 115, 6966, 8, 141, 1844, 5, 29894, 16, 116, 2]
<s>S: hmm then maybe i can change it to a freedive how long it would take to freedive but then it also depends on the thrust </s>T: since its a free dive, normal people obviously wont survive becasue of the lack of air. so we can also factor out the water pressure because that would obviously kill us too. so wouldnt it just be a matter of how fast this guy could swim and how deep the 

## 4. Define a model

In [ ]:
# define a model for masked language modeling
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)

model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

## 5. Define a trainer

In [ ]:
# define data_collator for masked language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15, # masking proportion
)

In [ ]:
with PRETRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
  config = yaml.safe_load(f)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_OUT_DIR,
    seed=SEED,
    **config["training_args"],
)

trainer = Trainer(

    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 6. Train the model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.173569,2.019144
2,2.025233,1.962911
3,1.938276,1.881902


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=10875, training_loss=2.1048156092852013, metrics={'train_runtime': 1269.5735, 'train_samples_per_second': 274.105, 'train_steps_per_second': 8.566, 'total_flos': 4.108916026906534e+16, 'train_loss': 2.1048156092852013, 'epoch': 3.0})

7. Save the model

In [ ]:
trainer.save_model(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./further_pretraining_model/further_pretrainng__preceding_1_turn__with_speaker_info/best_model/tokenizer_config.json',
 './further_pretraining_model/further_pretrainng__preceding_1_turn__with_speaker_info/best_model/tokenizer.json')

In [ ]:
from google.colab import runtime
runtime.unassign()